# Laboratorio 02 â€” Auto Loader y Calidad de Datos en Lakeflow

**Semana:** 05 | **Actividad de referencia:** Actividad 02  
**Modalidad:** Individual | **Entorno:** Databricks Lakeflow (Spark Declarative Pipelines)

---

## Instrucciones generales

Implementa una pipeline con Auto Loader (`cloudFiles`) para procesar tu dataset de forma incremental, y agrega expectativas de calidad (`@dp.expect`, `@dp.expect_or_drop`, `@dp.expect_or_fail`) con la estrategia de quarantine para separar registros invÃ¡lidos.

## Parte 1 â€” DescripciÃ³n del dataset y diseÃ±o de calidad

1. **Nombre, fuente y URL** del dataset.
2. **Reglas de calidad:** Define al menos 4 expectativas. Para cada una indica:
   - Nombre de la expectativa
   - ExpresiÃ³n SQL de la condiciÃ³n
   - AcciÃ³n: `expect` (warn), `expect_or_drop` (quarantine), `expect_or_fail` (detener pipeline)
   - JustificaciÃ³n de negocio
3. **Estrategia de quarantine:** Â¿QuÃ© harÃ¡s con los registros que fallen las expectativas `_or_drop`?

**Escribe tu respuesta aquÃ­:**

| Expectativa | ExpresiÃ³n | AcciÃ³n | JustificaciÃ³n |
|---|---|---|---|
| id_no_nulo | `id IS NOT NULL` | expect_or_drop | Sin ID no se puede rastrear |
| monto_positivo | `monto > 0` | expect_or_drop | Montos negativos indican error de sistema |
| categoria_valida | `categoria IN ('A', 'B', 'C')` | expect | Categoria desconocida â†’ solo advertencia |
| fecha_reciente | `fecha >= '2020-01-01'` | expect | Datos antiguos son sospechosos |

## Parte 2 â€” Importaciones y parÃ¡metros de pipeline

In [ ]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

# ParÃ¡metros de pipeline â€” configÃºralos en Lakeflow Settings â†’ Parameters
RUTA_LANDING     = spark.conf.get("ruta_landing",     "/Volumes/workspace/default/week_5/landing/")
SCHEMA_LOCATION  = spark.conf.get("schema_location",  "/Volumes/workspace/default/week_5/schema_hints/")
ENTORNO          = spark.conf.get("entorno",           "dev")

print(f"ruta_landing    = {RUTA_LANDING}")
print(f"schema_location = {SCHEMA_LOCATION}")
print(f"entorno         = {ENTORNO}")

## Parte 3 â€” Perfil tÃ©cnico del dataset (exploraciÃ³n interactiva previa)

In [ ]:
# Ejecutar interactivamente (fuera de la pipeline) para diseÃ±ar las expectativas
df_preview = spark.read.format("csv") \
    .option("header", True) \
    .option("inferSchema", True) \
    .load(f"{RUTA_LANDING}/*.csv")

total = df_preview.count()
print(f"Total registros: {total:,} | Columnas: {len(df_preview.columns)}")
df_preview.printSchema()
df_preview.show(5, truncate=False)

In [ ]:
# Simular cuÃ¡ntos registros fallarÃ­an cada expectativa
# Ajusta las columnas y condiciones segÃºn tu dataset real
from pyspark.sql import functions as F

reglas = {
    "id_no_nulo":       "columna_id IS NOT NULL",
    "monto_positivo":   "columna_numerica > 0",
    "categoria_valida": "columna_categoria IS NOT NULL",
}

print(f"{'Expectativa':<25} {'Fallos':>10} {'%':>8}")
print("-" * 45)
for nombre, condicion in reglas.items():
    fallos = df_preview.filter(f"NOT ({condicion})").count()
    pct    = round(fallos * 100.0 / total, 1)
    print(f"{nombre:<25} {fallos:>10,} {pct:>8}%")

**AnÃ¡lisis:** Â¿CuÃ¡les expectativas drop van a afectar mÃ¡s al conteo final de Silver? Â¿Son aceptables esas pÃ©rdidas?

## Parte 4 â€” Pipeline con Auto Loader y Expectativas

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Bronze: ingestiÃ³n incremental con Auto Loader (cloudFiles)
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@dp.table(
    comment="Bronze raw: Auto Loader lee archivos CSV del directorio landing de forma incremental."
)
def bronze_autoloader_mi_dataset():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format",         "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
        .option("header",                    "true")
        .option("inferSchema",               "true")
        .load(RUTA_LANDING)
        .withColumn("_source_file",   F.input_file_name())
        .withColumn("_ingest_ts",     F.current_timestamp())
        .withColumn("_entorno",       F.lit(ENTORNO))
    )

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Silver con expectativas de calidad
# Ajusta las columnas y condiciones a tu dataset real
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@dp.table(
    comment="Silver: registros que pasan todas las expectativas de calidad."
)
@dp.expect_or_drop("id_no_nulo",     "columna_id IS NOT NULL")
@dp.expect_or_drop("monto_positivo", "columna_numerica > 0")
@dp.expect("categoria_valida",       "columna_categoria IS NOT NULL")  # solo warn
def silver_mi_dataset_calidad():
    return dp.read_stream("bronze_autoloader_mi_dataset") \
        .withColumn("columna_texto", F.trim(F.lower(F.col("columna_texto"))))
        # AÃ±ade aquÃ­ las transformaciones Silver que correspondan

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Quarantine: registros que fallaron alguna expectativa _or_drop
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@dp.table(
    comment="Quarantine: registros rechazados por expectativas, para anÃ¡lisis y correcciÃ³n."
)
def quarantine_mi_dataset():
    return (
        dp.read_stream("bronze_autoloader_mi_dataset")
        .filter(
            F.col("columna_id").isNull() |
            (F.col("columna_numerica") <= 0)
        )
        .withColumn("_quarantine_ts",    F.current_timestamp())
        .withColumn("_razon_rechazo",    F.lit("Fallo expectativa id_no_nulo o monto_positivo"))
    )

In [ ]:
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Gold: agregaciÃ³n sobre Silver limpio
# â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
@dp.table(
    comment="Gold: KPIs calculados sobre datos Silver de calidad validada."
)
def gold_resumen_mi_dataset():
    return (
        dp.read("silver_mi_dataset_calidad")
        .groupBy("columna_categorica")
        .agg(
            F.count("*").alias("total"),
            F.avg("columna_numerica").alias("promedio"),
            F.sum("columna_numerica").alias("suma_total")
        )
    )

## Parte 5 â€” Verificar resultados despuÃ©s de ejecutar la pipeline

> Ejecuta estas celdas en un notebook interactivo separado DESPUÃ‰S de correr la pipeline en Lakeflow.

In [ ]:
# Comparar Bronze â†’ Silver â†’ Quarantine
# Ajusta los nombres de tabla si cambiaste el 'target' del pipeline
tables = [
    "workspace.default.bronze_autoloader_mi_dataset",
    "workspace.default.silver_mi_dataset_calidad",
    "workspace.default.quarantine_mi_dataset",
    "workspace.default.gold_resumen_mi_dataset"
]

for t in tables:
    try:
        cnt = spark.table(t).count()
        print(f"  {t.split('.')[-1]:<45} {cnt:>10,} filas")
    except Exception as e:
        print(f"  {t.split('.')[-1]:<45} ERROR: {e}")

In [ ]:
# Analizar quarantine: Â¿quÃ© reglas fallaron mÃ¡s?
spark.table("workspace.default.quarantine_mi_dataset") \
    .groupBy("_razon_rechazo") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(truncate=False)

## Parte 6 â€” Preguntas de negocio

1. Â¿QuÃ© porcentaje de los datos Bronze llegÃ³ a Silver? Â¿Es aceptable para tu caso de uso?
2. Â¿Los datos en Quarantine tienen algÃºn patrÃ³n (todos del mismo archivo fuente, misma fecha, misma categorÃ­a)?
3. Â¿QuÃ© acciÃ³n correctiva tomarÃ­as con los registros en Quarantine?
4. Â¿QuÃ© hallazgo de negocio muestra la tabla Gold?

## Parte 7 â€” ReflexiÃ³n final

1. Â¿QuÃ© ventaja tiene Auto Loader sobre una lectura batch normal con `spark.read`?
2. Â¿En quÃ© caso usarÃ­as `@dp.expect_or_fail` en lugar de `@dp.expect_or_drop`?
3. Â¿CÃ³mo agregarÃ­as un alerta (e.g. notificaciÃ³n por email o webhook) si el porcentaje de quarantine supera el 10%?
4. Â¿CÃ³mo verÃ­as las mÃ©tricas de expectativas en la UI de Lakeflow despuÃ©s de ejecutar la pipeline?

---

## Entrega en Git

```bash
# Copia el template a tu carpeta (solo la primera vez)
# cp semana_05/laboratorios/lab_02_autoloader_quality.ipynb semana_05/laboratorios/<tu-nombre>/lab_02_autoloader_quality.ipynb

git add semana_05/laboratorios/<tu-nombre>/lab_02_autoloader_quality.ipynb
git commit -m "lab: semana05 lab02 auto loader expectativas quarantine <nombre-dataset> - <tu-nombre>"
git push origin develop
```